# Preprocess MCNS Tables

One-time Male CNS preprocessing runbook. Raw table choices come from [`paths.py`](../../../src/cex/dataset/paths.py), processing logic lives in [`raw.py`](../../../src/cex/preprocessing/raw.py), and normalized output names live in [`schema.py`](../../../src/cex/dataset/schema.py). Set `DATASET_NAME` to `"mcns"` or `"mcns_v1"` for the corresponding release.


In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path

import numpy as np

import cex
from cex import get_dataset
import cex.dataset.paths as dpaths
import cex.dataset.schema as dsch
import cex.preprocessing as dprep
import cex.preprocessing.io_stat as io_stat
from cex.util import minimize_int_dtype
import cex.util.tables as rt

DATASET_NAME = "mcns"
REPO_ROOT = Path(cex.__file__).resolve().parents[2]
DATA_ROOT = REPO_ROOT / "data" / DATASET_NAME
WRITE_PREPROCESSED_Q = True
MCNS_SYN_VOXEL_SIZE_NM = 8

mcns = get_dataset(DATASET_NAME, DATA_ROOT)
fp_download = mcns.paths.fp_download
fp_preprocess = mcns.paths.fp_preprocessed
mcns_files = {name: os.path.join(fp_download, fn) for name, fn in dpaths.download_files_for_dataset(mcns.name).items()}
fp_download, fp_preprocess


## Optional Download

Set `DOWNLOAD_URL` after placing the dataset archive in Dropbox. The archive is downloaded to `download_data` and extracted there. Leave it as `None` when the raw files are already present.


In [ ]:
DOWNLOAD_URL = None
DOWNLOAD_ARCHIVE_NAME = None
DOWNLOAD_OVERWRITE_Q = False

if DOWNLOAD_URL is not None:
    dprep.download_and_extract(DOWNLOAD_URL, fp_download, DOWNLOAD_ARCHIVE_NAME, DOWNLOAD_OVERWRITE_Q)


## Inspect Downloaded Tables


In [ ]:
for name, fp in mcns_files.items():
    print(f"\n{name}:")
    print(rt.table_columns(fp))
    # display(rt.preview_table(fp, n=2))


## Convert `cell_data.parquet`


In [ ]:
MCNS_BODY_ANNOTATION_COLUMNS = None
if MCNS_BODY_ANNOTATION_COLUMNS is None:
    MCNS_BODY_ANNOTATION_COLUMNS = dpaths.MCNS_BODY_ANNOTATION_COLUMNS
print(f"Using MCNS body annotation columns: {MCNS_BODY_ANNOTATION_COLUMNS}")

In [ ]:
# combine cell data 
cell_data = rt.read_table(mcns_files["body_annotations"], columns=list(MCNS_BODY_ANNOTATION_COLUMNS))
cell_transmitter_data = rt.read_table(mcns_files["body_neurotransmitters"])
# name normalization
mcns_body_annotation_name_map = {
    'bodyId': 'rid', 'assignedOlHex1': 'p', 'assignedOlHex2': 'q', 
    'somaSide': 'side', 'flywireType': 'flywire_type',
}
mcns_body_transmitter_name_map = {'body': 'rid', 'consensus_nt': 'nt', 'cell_type': 'type'}

cell_transmitter_data.rename(columns=mcns_body_transmitter_name_map, inplace=True)
cell_transmitter_data.drop(columns=['type'], inplace=True)
cell_data.rename(columns=mcns_body_annotation_name_map, inplace=True)
cell_data = cell_data.merge(cell_transmitter_data, on="rid", how="left")
# preprocess data
cell_data = dprep.sort_table_and_add_id(cell_data, sort_cols=["type", "rid"], id_col_name="id")
cell_data["side"] = dprep.normalize_side_values(cell_data["side"])

cell_data_save_cols = ['id', 'rid', 'type', 'side']
if WRITE_PREPROCESSED_Q:
    rt.write_table(cell_data[cell_data_save_cols], 
                   os.path.join(fp_preprocess, dsch.CELL_DATA_FILE))
cell_data.head(2)

## Convert `type_data.parquet`

In [ ]:
mcns_nt_col = 'nt'
mcns_valud_col = ['nt', 'superclass', 'flywire_type']

cell_data[mcns_nt_col] = dprep.normalized_categorical_values(cell_data[mcns_nt_col], 
                                dict_map=dsch.NEUROTRANSMITTER_SYNONYMS, 
                                default_val=dsch.DEFAULT_UNKNOWN).astype(str)
type_data = dprep.build_type_data(cell_data, value_cols=mcns_valud_col, 
                                  nt_correction=dsch.TYPE_NEUROTRANSMITTER_GT)

if WRITE_PREPROCESSED_Q:
    rt.write_table(type_data, os.path.join(fp_preprocess, dsch.TYPE_DATA_FILE))
type_data.head()

## Convert `columns_data.npz`


In [ ]:
is_col_cell_Q = (cell_data['p'].notna()) & (cell_data['q'].notna())
column_data = {k: minimize_int_dtype(v.values.astype(np.int64), allow_bool=False)
               for k, v in cell_data[is_col_cell_Q].items() \
               if k in ['id', 'side', 'p', 'q']}
column_data['pq_min'] = np.asarray([column_data['p'].min(), column_data['q'].min()], dtype=np.int16)
column_data['pq_max'] = np.asarray([column_data['p'].max(), column_data['q'].max()], dtype=np.int16)

if WRITE_PREPROCESSED_Q:
    rt.write_table(column_data, os.path.join(fp_preprocess, dsch.COLUMN_DATA_FILE))

column_data

## Convert `synapses.parquet`

[`build_mcns_synapse_source_data`](../../../src/cex/preprocessing/raw.py) ports the partner-table center-coordinate conversion from the private MCNS preprocessing workflow before normalized ID lookup.


In [ ]:
synapse_partner_table = rt.read_table(mcns_files["syn_partners"],
    columns=list(dpaths.MCNS_SYNAPSE_PARTNER_COLUMNS))

In [ ]:
synapse_source_table = dprep.build_mcns_synapse_source_data(synapse_partner_table, 
                                                            voxel_size_nm=MCNS_SYN_VOXEL_SIZE_NM)

In [ ]:
synapse_table = dprep.normalize_synapse_table(synapse_source_table, cell_data, 
                                              pre_rid_col='pre_root_id', 
                                              post_rid_col='post_root_id', 
                                              x_col='ctr_x', y_col='ctr_y', z_col='ctr_z',
                                              require_known_Q=False,
                                              keep_unknown_synapses_Q=False)

In [ ]:
if WRITE_PREPROCESSED_Q:
    rt.write_table(synapse_table, os.path.join(fp_preprocess, dsch.SYNAPSE_DATA_FILE))

synapse_table.head()


## Convert `cell_to_cell_syn_count.parquet`


In [ ]:
connectivity = dprep.build_connectivity_edges(synapse_table)

if WRITE_PREPROCESSED_Q:
    rt.write_table(connectivity, os.path.join(fp_preprocess, dsch.CELL_TO_CELL_SYN_COUNT_FILE))

connectivity.head()


## Aggregate Type Connectivity


In [ ]:
type_connectivity_data = dprep.build_type_connectivity_data(connectivity, cell_data, type_data["type"].values)
type_connectivity_fp = os.path.join(fp_preprocess, dsch.TYPE_TO_TYPE_SYN_COUNT_FILE)
if WRITE_PREPROCESSED_Q:
    rt.write_table(type_connectivity_data, type_connectivity_fp)
type_connectivity_fp, type_connectivity_data["shape"]


## Per-Cell IO Statistics

In [ ]:
RUN_IO_STAT_Q = True
IO_STAT_NUM_WORKERS = 8

if RUN_IO_STAT_Q:
    io_stat.compute_all_io_stats(
        mcns,
        stat_type=dpaths.DEFAULT_IO_STAT_TYPE,
        min_syn_per_rid=1,
        min_frac=1e-2,
        num_workers=IO_STAT_NUM_WORKERS,
        write_Q=True,
    )

In [ ]:
RUN_COMBINED_PHOTORECEPTOR_IO_STAT_Q = True
COMBINED_PHOTORECEPTOR_TYPES = ["R7", "R8"]

if RUN_COMBINED_PHOTORECEPTOR_IO_STAT_Q:
    for cell_type in COMBINED_PHOTORECEPTOR_TYPES:
        tables = io_stat.compute_cell_type_io_stat(
            cell_type,
            mcns,
            stat_type=dpaths.DEFAULT_IO_STAT_TYPE,
            remove_autapse_Q=True,
            min_syn_per_rid=1,
            min_frac=1e-2,
            write_Q=True,
        )
        print(cell_type, {direction: table.shape[0] for direction, table in tables.items()})

In [ ]:
mcns.show_cell_type_io_stat("R8", 'output')